In [ ]:
import itertools
import stim
from IPython.display import Markdown

from library.qubit_array import QubitArray
from library.steane_code.patch import SteaneCodePatch
from utils.error_rate_analyser import simulate

In [ ]:
cumulative = stim.Circuit()
array = QubitArray(cumulative, dimensions = (5, 3))

steane = SteaneCodePatch(array)

scenarios: dict[str, stim.Circuit] = dict()
point = 0

In [ ]:
# Generate circuit up to and including preparation (w/ S-injection)
steane.append_preparation(cumulative)

raw = cumulative.copy()
steane.append_observable(raw, observable='Y', support = steane.logical)
scenarios[f'Point {point} - Preparation'] = raw
point += 1

display(Markdown(f"[Open in Crumble (Raw)]({raw.to_crumble_url()})"))

In [ ]:
def annotate_detectors(circuit: stim.Circuit, sdc_rounds: int = 0):
    for color in steane.stabilizers.keys():
        if sdc_rounds >= 1:
            steane.annotate_detector(circuit, f"SDC0:X{color}")
            steane.annotate_detector(circuit, f"SDC0:Z{color}")
        for curr, next in itertools.pairwise(range(sdc_rounds)):
            steane.annotate_detector(circuit, f"SDC{curr}:Z{color}", f"SDC{next}:Z{color}")

    for curr, next in itertools.pairwise(range(sdc_rounds)):
        steane.annotate_detector(circuit, f"SDC{next}:XG", f"SDC{curr}:XR", f"SDC{curr}:XG")
        steane.annotate_detector(circuit, f"SDC{next}:XB", f"SDC{curr}:XG")
        steane.annotate_detector(circuit, f"SDC{next}:XR")

    for measurement in range(6):
        steane.annotate_detector(circuit, f"CULT:X{measurement}")

In [ ]:
# Generate circuit up to and including superdense code cycles
for s in range(3):
    label = f"SDC{s}"
    steane.append_superdense(cumulative, prefix=label)

    sdc = cumulative.copy()
    annotate_detectors(sdc, s+1)
    steane.append_observable(sdc, observable='Y', support = steane.logical)

    scenarios[f'Point {point} - {label}'] = sdc

    missing = len(sdc.missing_detectors())
    warning = f"[Missing detectors : {missing}]" if missing > 0 else ""
    display(Markdown(f"[Open in Crumble (SDCx{s+1})]({sdc.to_crumble_url()}) {warning}"))
    point += 1

In [ ]:
# Generate circuit up to and including double-check-S
for c in range(1):
    steane.append_cultivation(cumulative, prefix="CULT")

    dcs = cumulative.copy()
    annotate_detectors(dcs, sdc_rounds=3)
    steane.append_observable(dcs, observable='Y', support = steane.logical)

    scenarios[f'Point {point} - Double-Check-S'] = dcs
    point += 1

    missing = len(dcs.missing_detectors())
    warning = f"[Missing detectors : {missing}]" if missing > 0 else ""
    display(Markdown(f"[Open in Crumble (DCSx{c+1})]({dcs.to_crumble_url()}) {warning}"))

In [ ]:
# Analyse error rates of all cumulative circuits
title = r"Steane Code $|\mathbf{S}\rangle$ [$\mathbf{Y}^{\otimes 7}$ observable]"
simulate(scenarios, title, postselection=True, shots=1e7, minimal_noise=-7, figsize=(11, 4.5))